# Getting Started with Strands Agents

## Overview
Strands Agents is a powerful framework for building AI agents that can interact with AWS services and perform complex tasks. This notebook walks you through creating your first Strands agent, from a simple agent to one that uses custom tools, and finishes with a practical RecipeBot use case.

## Tutorial Details

| Information          | Details                              |
|:---------------------|:-------------------------------------|
| Agent structure      | Single agent                         |
| Model provider       | Amazon Bedrock                       |
| Model                | Anthropic Claude Sonnet 4.5          |
| Native tools used    | none                                 |
| Custom tools created | calculator, weather, websearch |
| Strands features     | Agent, @tool decorator, BedrockModel |

## What you'll learn
* Create an agent and give it a system prompt
* Add custom tools
* Invoke a tool directly with `agent.tool.<name>`
* Configure logging
* Choose and configure a model provider

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account
* Anthropic Claude Sonnet 4.5 enabled on Amazon Bedrock, [guide](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)

Let's now install the required packages for our agent.

In [ ]:
# Install Strands using pip

!pip install strands-agents


## Creating Your First Agent

Let's get an overview of the agentic components needed.

### Create a simple agent

This will create an agent with the default model provider, [Amazon Bedrock](https://aws.amazon.com/bedrock/). If you do not pass a `model`, the agent uses the default Amazon Bedrock model (covered in the Model provider section below). While the agent runs in your local environment, the Amazon Bedrock model runs in the cloud, and your agent invokes it there. The architecture looks as follows:

<div style="text-align:center">
    <img src="images/simple_agent.png" width="75%" />
</div>

In [ ]:
import warnings
warnings.filterwarnings(action="ignore", message=r"datetime.datetime.utcnow") 

from strands import Agent
# Initialize your agent
agent = Agent(
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",  # Optional: Specify the model ID
    system_prompt="You are a helpful assistant that provides concise responses."
)

# Send a message to the agent
response = agent("Hello! Tell me a joke.")

### Add tools to the agent

You create custom tools using the `@tool` decorator. The [strands-agents-tools](https://github.com/strands-agents/tools) repository also provides pre-built tools that you can import. The function's typed arguments and docstring are not just documentation. Strands turns them into the tool's input schema, which is what the model reads to decide when and how to call the tool. We can create agents with custom tools. For instance, defining a small calculator tool and a custom tool for getting the weather, you get the following architecture:
<div style="text-align:center">
    <img src="images/agent_with_tools.png" width="75%" />
</div>

Implementing this architecture, you have the following:

In [ ]:
import operator
from typing import Literal

from strands import Agent, tool

_OPS = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": operator.truediv,
    "**": operator.pow,
}


@tool
def calculator(a: float, b: float, op: Literal["+", "-", "*", "/", "**"]) -> float:
    """Apply an arithmetic operator to two numbers.

    Args:
        a: Left operand.
        b: Right operand.
        op: One of "+", "-", "*", "/", "**".
    """
    return _OPS[op](a, b)

# Create a custom tool
@tool
def weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: Name of the city to get the weather for.
    """
    # Dummy implementation - a real tool would call a weather API
    return f"It is sunny and 72°F in {city}."

agent = Agent(
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather.")

response = agent("What is the weather in Seattle today?")
print(response)

### Invoking tool directly

For some applications it is important to directly call the tool. For instance, you might want to debug the tool, pre-populate the agent knowledge with your customer's information, or use a tool inside another tool. In Strands you can do it using the ``tool`` method of your agent followed by the tool name.

In [ ]:
# Alternatively, you can invoke the tool directly like so:
agent.tool.calculator(a=144, b=0.5, op="**")


### Change the log level and format

Strands Agents SDK uses Python's standard `logging` module to provide visibility into its operations.

The Strands Agents SDK implements a straightforward logging approach:

1. **Module-level Loggers**: Each module in the SDK creates its own logger using logging.getLogger(__name__), following Python best practices for hierarchical logging.
2. **Root Logger**: All loggers in the SDK are children of the "strands" root logger, making it easy to configure logging for the entire SDK.
3. **Default Behavior**: By default, the SDK doesn't configure any handlers or log levels, allowing you to integrate it with your application's logging configuration.

To enable logging for the Strands Agents SDK, you can configure the **"strands"** logger. If you want to change the log level, for example during debugging, or modify the log format, you can set the logger configuration as follows:

In [ ]:
import logging
from strands import Agent

# Enables Strands debug log level
logging.getLogger("strands").setLevel(logging.DEBUG) # or logging.INFO

# Sets the logging format and streams logs to stderr
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()]
)

agent = Agent(model="us.anthropic.claude-sonnet-4-5-20250929-v1:0")
agent("Hello!")


### Model provider

The default model provider is [Amazon Bedrock](https://aws.amazon.com/bedrock/). If you do not pass a `model`, the agent uses the SDK's default Bedrock model. Your AWS region selects the Bedrock endpoint your agent calls, not which default model it gets.

You can specify a different model in Amazon Bedrock providing the model ID string directly:

In [ ]:
from strands import Agent

agent = Agent(model="us.anthropic.claude-haiku-4-5-20251001-v1:0")
print(agent.model.config)


For more control over the model configuration, you can create a `BedrockModel` provider instance:

In [ ]:
import boto3
from strands import Agent
from strands.models import BedrockModel

# Create a BedrockModel
bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    region_name='us-west-2',
    temperature=0.3,
)

agent = Agent(model=bedrock_model)


More details on the available model providers on the [Model Provider Quickstart page](https://strandsagents.com/docs/user-guide/quickstart/python/#model-providers)


## [Optional] Let's Build a Task-Specific Agent - RecipeBot

Let's create a practical example of a task-specific agent. We create a `RecipeBot` that recommends recipes and answers any cooking-related questions.

Here's what we will create:

<div style="text-align:center">
    <img src="images/interactive_recipe_agent.png" width="75%" />
</div>

In [ ]:
# Install ddgs for the RecipeBot web search tool
%pip install ddgs

In [ ]:
from strands import Agent, tool
from ddgs import DDGS
from ddgs.exceptions import RatelimitException, DDGSException
import logging

# Configure logging
logging.getLogger("strands").setLevel(logging.INFO)

# Define a websearch tool
@tool
def websearch(keywords: str, region: str = "us-en", max_results: int | None = None) -> str:
    """Search the web to get updated information.
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "RatelimitException: Please try again after a short delay."
    except DDGSException as d:
        return f"DuckDuckGoSearchException: {d}"
    except Exception as e:
        return f"Exception: {e}"


In [ ]:
# Create a recipe assistant agent
recipe_agent = Agent(
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt="""You are RecipeBot, a helpful cooking assistant.
    Help users find recipes based on ingredients and answer cooking questions.
    Use the websearch tool to find recipes when users mention ingredients or to
    look up cooking information.""",
    # Import the websearch tool we created above
    tools=[websearch],
)

In [ ]:
response = recipe_agent("Suggest a recipe with chicken and broccoli.")

# Other examples:
# response = recipe_agent("How do I cook quinoa?")
# response = recipe_agent("How can I substitute white wine in shrimp pasta?")
# response = recipe_agent("What are the health benefits of asparagus?")
print(response)

For more detail, check out the [Strands documentation](https://strandsagents.com/docs/user-guide/quickstart/python/).

### [Optional] Run RecipeBot via CLI

You can run the agent in interactive mode via the command line (for instance using the terminal on SageMaker Studio) through the python script provided in `recipe-bot-cli/recipe_bot.py`. This allows you to interact with the agent in a more dynamic way, sending messages and receiving responses via the CLI.
Run these commands on a command line interface to run the agent in interactive mode:

```bash
cd samples/python/01-learn/01-first-agent/recipe-bot-cli/
pip install -r requirements.txt
python recipe_bot.py
```

With this, you can talk to the bot via a command line interface (CLI).

## Congratulations!

In this notebook you built your first Strands agent, added custom tools, invoked tools directly, configured logging, chose a model provider, and assembled a RecipeBot. Next, let's look at adding tools and connecting MCP servers.